In [21]:
from api_query import get_wikitext
import re

Title = "Liste der Vögel Deutschlands"
data = get_wikitext(Title)

In [22]:
print(data)

Die '''Liste der Vögel Deutschlands''' führt die Arten der [[Vögel]] (Aves) auf, deren Vorkommen in [[Deutschland]] beobachtet worden sind.

Sie enthält nur Arten, deren Feststellung von den zuständigen ornithologischen Gremien (zum Beispiel durch die Deutsche Avifaunistische Kommission (DAK)<ref>[https://www.dda-web.de/dak Dachverband Deutscher Avifaunisten – Über die DAK], abgerufen am 17. Dezember 2022</ref> oder entsprechende Einrichtungen auf Länderebene) anerkannt und in einschlägigen Fachzeitschriften publiziert wurden. Außerdem werden an dieser Stelle nur die Arten berücksichtigt, deren Beobachtung auf Feststellung wildlebender Tiere zurückzuführen ist sowie diejenigen, die eingebürgert sind, das heißt, die ursprünglich in anderen Regionen vorkommen, die sich jedoch inzwischen durch Fortpflanzung längerfristig etabliert haben.

Grundlage für die nachfolgende Liste ist die ''Liste der Vögel Deutschlands – Version 3.2'' mit dem Stand vom 30. Juni 2019, wie sie von Peter H. Barthe

In [23]:
queries = [i for i in data.split(sep="==")]

In [24]:
hits = list()
for query in queries:
    hits.append(re.findall(pattern=r"^^\|\[\[Datei:[^\]]*\]\]\|\|\[\[([^\[\]|]+)\]\]\|\|''([^']+)''\|\|([^|]*)\|\|([^|]*)", string = query, flags=re.MULTILINE))

In [25]:
import itertools
german_birds = list(itertools.chain.from_iterable(hits))



In [26]:
import pandas as pd

birds_df = pd.DataFrame(german_birds)


In [27]:
birds_df.columns = ["German_name", "Scientific_name", "Status", "Rl-status"]


In [28]:
birds_df.set_index("Scientific_name")

,German_name,Status,Rl-status
Scientific_name,,,
Tetrastes bonasia,Haselhuhn,"Brut-, Jahresvogel",2
Tetrao urogallus,Auerhuhn,"Brut-, Jahresvogel",1
Lyrurus tetrix,Birkhuhn,"Brut-, Jahresvogel",2
Lagopus muta,Alpenschneehuhn,"Brut-, Jahresvogel",R
Alectoris graeca,Steinhuhn,"seltener, lokaler Brut-, Jahresvogel",R
...,...,...,...
Emberiza bruniceps,Braunkopfammer,Ausnahmeerscheinung,
Emberiza spodocephala,Maskenammer,besondere Ausnahmeerscheinung,
Emberiza schoeniclus,Rohrammer,"Brut-, Zugvogel, Wintergast",


In [29]:
import pandas as pd

taxonomy = pd.read_csv("birds_taxonomy.csv")
taxonomy

,Unnamed: 0,Taxon_rank,Order,Family,Family_English_name,Scientific_name,English_name_AviList
0,0,order,Struthioniformes,NaN,NaN,Struthioniformes,NaN
1,1,family,Struthioniformes,Struthionidae,Ostriches,Struthionidae,NaN
2,2,genus,Struthioniformes,Struthionidae,Ostriches,Struthio,NaN
3,3,species,Struthioniformes,Struthionidae,Ostriches,Struthio molybdophanes,Somali Ostrich
4,4,species,Struthioniformes,Struthionidae,Ostriches,Struthio camelus,Common Ostrich
...,...,...,...,...,...,...,...
33679,33679,subspecies,Passeriformes,Thraupidae,Tanagers and Allies,Stilpnia cyanicollis cyanicollis,NaN
33680,33680,subspecies,Passeriformes,Thraupidae,Tanagers and Allies,Stilpnia cyanicollis cyanopygia,NaN
33681,33681,subspecies,Passeriformes,Thraupidae,Tanagers and Allies,Stilpnia cyanicollis hannahiae,NaN
33682,33682,subspecies,Passeriformes,Thraupidae,Tanagers and Allies,Stilpnia cyanicollis melanogaster,NaN


In [30]:
birds_df.dtypes

German_name        str
Scientific_name    str
Status             str
Rl-status          str
dtype: object

In [31]:
taxonomy.dtypes

Unnamed: 0              int64
Taxon_rank                str
Order                     str
Family                    str
Family_English_name       str
Scientific_name           str
English_name_AviList      str
dtype: object

In [32]:
df_final = taxonomy.join(other=birds_df.set_index("Scientific_name"),
                          on="Scientific_name",
                          how="inner")

In [33]:
df_final = df_final.drop(columns='Unnamed: 0').copy()

In [35]:
df_final.reset_index(drop=True, inplace=True)
df_final

,Taxon_rank,Order,Family,Family_English_name,Scientific_name,English_name_AviList,German_name,Status,Rl-status
0,species,Anseriformes,Anatidae,"Ducks, Swans, and Geese",Oxyura leucocephala,White-headed Duck,Weißkopf-Ruderente,Ausnahmeerscheinung,
1,species,Anseriformes,Anatidae,"Ducks, Swans, and Geese",Oxyura jamaicensis,Ruddy Duck,Schwarzkopf-Ruderente,seltener Jahresvogel,
2,species,Anseriformes,Anatidae,"Ducks, Swans, and Geese",Cygnus olor,Mute Swan,Höckerschwan,"Brut-, Jahres-, Zugvogel, Wintergast",
3,species,Anseriformes,Anatidae,"Ducks, Swans, and Geese",Cygnus cygnus,Whooper Swan,Singschwan,"seltener Brutvogel; Zugvogel, Wintergast",
4,species,Anseriformes,Anatidae,"Ducks, Swans, and Geese",Branta bernicla,Brant Goose,Ringelgans,"Zugvogel, Wintergast",
...,...,...,...,...,...,...,...,...,...
452,species,Passeriformes,Emberizidae,Old World Buntings,Emberiza caesia,Cretzschmar's Bunting,Grauortolan,besondere Ausnahmeerscheinung,
453,species,Passeriformes,Emberizidae,Old World Buntings,Emberiza leucocephalos,Pine Bunting,Fichtenammer,Ausnahmeerscheinung,
454,species,Passeriformes,Emberizidae,Old World Buntings,Emberiza citrinella,Yellowhammer,Goldammer,"Brut-, Jahres-, Zugvogel, Wintergast",
455,species,Passeriformes,Parulidae,New World Warblers,Setophaga americana,Northern Parula,Meisenwaldsänger,besondere Ausnahmeerscheinung,


In [38]:
df_final.loc[160:180]


,Taxon_rank,Order,Family,Family_English_name,Scientific_name,English_name_AviList,German_name,Status,Rl-status
160,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Sterna hirundo,Common Tern,Flussseeschwalbe,"Brut-, Zugvogel",2
161,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Sterna dougallii,Roseate Tern,Rosenseeschwalbe,Ausnahmeerscheinung,0
162,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Hydrocoloeus minutus,Little Gull,Zwergmöwe,"seltener, lokaler Brutvogel; Zugvogel",R
163,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Rhodostethia rosea,Ross's Gull,Rosenmöwe,Ausnahmeerscheinung,
164,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Rissa tridactyla,Black-legged Kittiwake,Dreizehenmöwe,"lokaler Brut-, Jahres-, Zugvogel, Wintergast",2
165,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Xema sabini,Sabine's Gull,Schwalbenmöwe,seltener Zugvogel,
166,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Pagophila eburnea,Ivory Gull,Elfenbeinmöwe,Ausnahmeerscheinung,
167,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Chroicocephalus genei,Slender-billed Gull,Dünnschnabelmöwe,besondere Ausnahmeerscheinung,
168,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Chroicocephalus philadelphia,Bonaparte's Gull,Bonapartemöwe,besondere Ausnahmeerscheinung,\n
169,species,Charadriiformes,Laridae,"Skimmers, Noddies, Terns, and Gulls",Chroicocephalus ridibundus,Black-headed Gull,Lachmöwe,"Brut-, Jahres-, Zugvogel, Wintergast",


In [36]:
df_final.to_csv("birds.csv")